# Cortex Functions: RBAC Runbook

**Purpose**: Examples and templates for implementing RBAC for Cortex Functions.

**Target Audience**: Snowflake Administrators

**Last Updated**: March 2026

**Version**: 2.0

---

## Overview

This runbook covers account-level model restrictions, role-based access control, model application roles, and fine-tuning with proper access management.

## 1. Configuration Variables

Set these variables at the beginning to customize the runbook for your environment.


In [ ]:
USE SECONDARY ROLES NONE;

-- 1.1 Configuration Variables
-- Modify these values according to your environment

SET DATABASE_NAME = 'RUNBOOKS_DB';
SET SCHEMA_NAME = 'CORTEX_FUNCTIONS';
SET WAREHOUSE_NAME = 'COMPUTE_WH';

-- Role Names
SET ADMIN_ROLE = 'CORTEX_ADMIN_ROLE';
SET DATA_SCIENTIST_ROLE = 'CORTEX_DS_ROLE';
SET ANALYST_ROLE = 'CORTEX_ANALYST_ROLE';
SET RESTRICTED_ROLE = 'CORTEX_RESTRICTED_ROLE';


-- Display configuration
SELECT 
    'Configuration Set' AS STATUS,
    $DATABASE_NAME AS DATABASE_NAME,
    $SCHEMA_NAME AS SCHEMA_NAME,
    $WAREHOUSE_NAME AS WAREHOUSE_NAME,
    $ADMIN_ROLE AS ADMIN_ROLE,
    $DATA_SCIENTIST_ROLE AS DATA_SCIENTIST_ROLE,
    $ANALYST_ROLE AS ANALYST_ROLE;


## 2. Account-Level Access Controls

Account-level controls allow you to manage Cortex Functions access across your entire Snowflake account.


### 2.1 Cross-Region Inferencing

Cross-region inferencing allows Cortex Functions to use models hosted in different regions. This is useful when specific models are not available in your home region.

**Note**: Cross-region inferencing may incur additional data transfer costs and latency.


In [ ]:
-- 2.1.1 Check current cross-region inferencing setting
USE SECONDARY ROLES NONE; -- Disable secondary roles for testing purposes

USE ROLE ACCOUNTADMIN;

SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;


In [ ]:
-- 2.1.2 Enable cross-region inferencing (requires ACCOUNTADMIN)
USE ROLE ACCOUNTADMIN;

ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;


When enabled, cross-region inference occurs if the LLM or feature is not supported in your default region.

By default, the parameter is set to DISABLED. This allows requests to be processed only in the default region. You can specify the regions you want to allow cross-region inference to using the ALTER ACCOUNT command.

More detail here: https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference

### 2.2 Model Allow List Management

Control which LLMs are available in your account. Values: `'ALL'`, `'NONE'`, or a comma-separated model list.

In [ ]:
-- 2.2.1 View and reset current allowed models
USE ROLE ACCOUNTADMIN;

ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'NONE'; --or ALL

SHOW PARAMETERS LIKE 'CORTEX_MODELS_ALLOWLIST' IN ACCOUNT;

In [ ]:
-- 2.2.2 Add specific models to the allow list
-- Available models: mistral-large, mistral-large2, llama3.1-8b, llama3.1-70b, llama3.1-405b, 
--                  reka-flash, mixtral-8x7b, gemma-7b, snowflake-arctic, etc.

USE ROLE ACCOUNTADMIN;

ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'mistral-large2,llama3.1-8b,llama3.1-70b';

SHOW PARAMETERS LIKE 'CORTEX_MODELS_ALLOWLIST' IN ACCOUNT;

## 3. Role-Based Access Controls (RBAC)

Implement fine-grained access control using Snowflake's RBAC model.


### 3.1 Setup: Create Database, Schema, and Warehouse


In [ ]:
-- 3.1.1 Create database, schema, and warehouse
USE ROLE ACCOUNTADMIN;

CREATE DATABASE IF NOT EXISTS IDENTIFIER($DATABASE_NAME);
USE DATABASE IDENTIFIER($DATABASE_NAME);
CREATE SCHEMA IF NOT EXISTS IDENTIFIER($SCHEMA_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);

-- Create warehouse for Cortex workloads
CREATE WAREHOUSE IF NOT EXISTS IDENTIFIER($WAREHOUSE_NAME)
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
    COMMENT = 'Warehouse for Cortex Functions';

### 3.2 Create Base Roles


In [ ]:
-- 3.2.1 Create custom roles for different access levels
USE ROLE ACCOUNTADMIN;

-- Admin role: Will have access to Cortex Functions and ability to create stored procedures 
-- containing AISQL with inherited caller rights (shared with restricted role)
CREATE ROLE IF NOT EXISTS IDENTIFIER($ADMIN_ROLE)
    COMMENT = 'Administrator role for Cortex Functions';
GRANT ALL ON DATABASE IDENTIFIER($DATABASE_NAME) TO ROLE IDENTIFIER($ADMIN_ROLE);
GRANT ALL ON SCHEMA IDENTIFIER($SCHEMA_NAME) TO ROLE IDENTIFIER($ADMIN_ROLE);
GRANT ALL ON WAREHOUSE IDENTIFIER($WAREHOUSE_NAME) TO ROLE IDENTIFIER($ADMIN_ROLE);


-- Data Scientist role: Will have access to all Cortex Functions including fine-tuning
CREATE ROLE IF NOT EXISTS IDENTIFIER($DATA_SCIENTIST_ROLE)
    COMMENT = 'Data Scientist role with full Cortex access';
GRANT USAGE ON DATABASE IDENTIFIER($DATABASE_NAME) TO ROLE IDENTIFIER($DATA_SCIENTIST_ROLE);
GRANT USAGE, CREATE TABLE, CREATE VIEW ON SCHEMA IDENTIFIER($SCHEMA_NAME) 
    TO ROLE IDENTIFIER($DATA_SCIENTIST_ROLE);
GRANT USAGE, OPERATE ON WAREHOUSE IDENTIFIER($WAREHOUSE_NAME) TO ROLE IDENTIFIER($DATA_SCIENTIST_ROLE);
GRANT CREATE MODEL ON SCHEMA IDENTIFIER($SCHEMA_NAME) TO ROLE IDENTIFIER($DATA_SCIENTIST_ROLE); -- For finetuned models

-- Analyst role: Can use basic Cortex Functions
CREATE ROLE IF NOT EXISTS IDENTIFIER($ANALYST_ROLE)
    COMMENT = 'Analyst role with limited Cortex access';
GRANT USAGE ON DATABASE IDENTIFIER($DATABASE_NAME) TO ROLE IDENTIFIER($ANALYST_ROLE);
GRANT USAGE ON SCHEMA IDENTIFIER($SCHEMA_NAME) TO ROLE IDENTIFIER($ANALYST_ROLE);
GRANT USAGE ON WAREHOUSE IDENTIFIER($WAREHOUSE_NAME) TO ROLE IDENTIFIER($ANALYST_ROLE);

-- Restricted role: Minimal access
CREATE ROLE IF NOT EXISTS IDENTIFIER($RESTRICTED_ROLE)
    COMMENT = 'Restricted role with minimal Cortex access';
GRANT USAGE ON DATABASE IDENTIFIER($DATABASE_NAME) TO ROLE IDENTIFIER($RESTRICTED_ROLE);
GRANT USAGE ON SCHEMA IDENTIFIER($SCHEMA_NAME) TO ROLE IDENTIFIER($RESTRICTED_ROLE);
GRANT USAGE ON WAREHOUSE IDENTIFIER($WAREHOUSE_NAME) TO ROLE IDENTIFIER($RESTRICTED_ROLE);

### 3.3 Cortex LLM Privileges

`SNOWFLAKE.CORTEX_USER` grants access to Cortex AISQL functions. By default it is granted to `PUBLIC`. Revoking it from `PUBLIC` and granting to specific roles restricts access.

To allow roles without `CORTEX_USER` to call Cortex Functions, use stored procedures with `EXECUTE AS OWNER` (demonstrated in Section 3.8).

In [ ]:
-- 3.3.1 Revoke Cortex User role from Public and assign to specific roles

-- Revoke from PUBLIC role
REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER
  FROM ROLE PUBLIC;

-- WARNING: This revokes ALL imported privileges from PUBLIC on the SNOWFLAKE database.
-- Only do this if you understand the impact on other features that depend on SNOWFLAKE database access.
REVOKE IMPORTED PRIVILEGES ON DATABASE SNOWFLAKE
  FROM ROLE PUBLIC;

-- Assign Cortex User role to specific roles
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE IDENTIFIER($ADMIN_ROLE);

GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE IDENTIFIER($DATA_SCIENTIST_ROLE);

GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE IDENTIFIER($ANALYST_ROLE);

-- Grant Cortex Admin role the rights to create stored procs with inherited caller rights

GRANT INHERITED CALLER USAGE ON ALL SCHEMAS IN DATABASE snowflake TO ROLE IDENTIFIER($ADMIN_ROLE);
GRANT INHERITED CALLER USAGE ON ALL FUNCTIONS IN DATABASE snowflake TO ROLE IDENTIFIER($ADMIN_ROLE);
GRANT CALLER USAGE ON DATABASE snowflake TO ROLE IDENTIFIER($ADMIN_ROLE);


-- Granting restricted role caller privileges: The restricted role needs to have caller rights on all objects that might be accessed through the stored procedure (alternatively by stored procedure).

### 3.4 Embedding Privileges

The CORTEX_EMBED_USER database role in the SNOWFLAKE database includes the privileges that allow users to call the text embedding functions AI_EMBED, EMBED_TEXT_768, and EMBED_TEXT_1024 and to create Cortex Search Services with managed vector embeddings.

In [ ]:
-- 3.4.1 Grant embedding privileges to Admin role
USE ROLE ACCOUNTADMIN;

GRANT DATABASE ROLE SNOWFLAKE.CORTEX_EMBED_USER TO ROLE IDENTIFIER($ADMIN_ROLE);

### 3.5 Model Application Roles 

Model application roles in `SNOWFLAKE.MODELS` let you control per-model access via RBAC. Grant the corresponding application role to allow a role to use a specific model beyond the allowlist.

In [ ]:
-- 3.5.1 Set model allowlist and view application roles
USE ROLE ACCOUNTADMIN;

-- By default, only this model will be available to users
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'mistral-large2';

-- Refresh base models (takes about 5 minutes to run - uncomment if needed)
-- CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();

-- View available Cortex Model Application Roles
SHOW APPLICATION ROLES IN APPLICATION SNOWFLAKE;


### 3.6 Create RBAC Hierarchy
Roles who have been granted the CORTEX_USER will by default only be able to call functions with LLM models that are on the allowlist (mistral large in above example). To expand, or assign grants on a per user/role basis you can leverage the respective model application roles.

In [ ]:
GRANT ROLE IDENTIFIER($RESTRICTED_ROLE) TO ROLE IDENTIFIER($ANALYST_ROLE);
GRANT ROLE IDENTIFIER($ANALYST_ROLE) TO ROLE IDENTIFIER($DATA_SCIENTIST_ROLE);
GRANT ROLE IDENTIFIER($DATA_SCIENTIST_ROLE) TO ROLE IDENTIFIER($ADMIN_ROLE);
GRANT ROLE IDENTIFIER($ADMIN_ROLE) TO ROLE SYSADMIN;


In [ ]:
-- 3.6.1 Grant model application role to Admin role
-- This allows the Admin role to use Claude 4 Sonnet in AISQL functions
USE ROLE ACCOUNTADMIN;

GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-CLAUDE-4-SONNET" TO ROLE IDENTIFIER($ADMIN_ROLE);

### 3.7 Test Function Usage with Different Roles

Test how different roles interact with Cortex Functions based on the allowlist and application role grants.


In [ ]:
-- 3.7.1 Test as Data Scientist Role
-- Should work with Model that's been added to allowlist (mistral-large-2 in above example)
USE ROLE IDENTIFIER($DATA_SCIENTIST_ROLE);
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);
USE WAREHOUSE IDENTIFIER($WAREHOUSE_NAME);

-- Test COMPLETE function
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'mistral-large2',
    'What are the key benefits of Snowflake Cortex?'
) AS response;

In [ ]:
-- 3.7.2 Test as Analyst Role
-- This should NOT work as we are attempting to use a model not on Allowlist
USE ROLE IDENTIFIER($ANALYST_ROLE);
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);
USE WAREHOUSE IDENTIFIER($WAREHOUSE_NAME);

-- Analysts can use basic Cortex functions
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'snowflake.models.claude-4-sonnet',
    'What are the key benefits of Snowflake Cortex?'
) AS response;

In [ ]:
-- 3.7.3 Test as Admin Role
-- This should work as we granted the admin role access to the respective application role
USE ROLE IDENTIFIER($ADMIN_ROLE);
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);
USE WAREHOUSE IDENTIFIER($WAREHOUSE_NAME);

-- Analysts can use basic Cortex functions
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'snowflake.models.claude-4-sonnet',
    'What are the key benefits of Snowflake Cortex?'
) AS response;

### 3.7.1 Limitations of Model RBAC

See here for more detail on features supported by model access controls: https://docs.snowflake.com/user-guide/snowflake-cortex/aisql#supported-features


### 3.8 Limited access to functions through stored procedures and owner's rights

You can grant users without the `CORTEX_USER` database role the ability to execute Cortex Functions through stored procedures with `EXECUTE AS OWNER`. The procedure runs with the owner's privileges, so the caller never needs direct Cortex access.

This example includes input validation (prompt length limit) and a system prompt to constrain behavior.

In [ ]:
-- 3.8.1 Create stored procedure as Admin Role (with CORTEX_USER privileges)
USE ROLE IDENTIFIER($ADMIN_ROLE);
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);

CREATE OR REPLACE PROCEDURE restricted_cortex_complete_claude(
    user_prompt VARCHAR
)
RETURNS VARCHAR
LANGUAGE SQL
EXECUTE AS OWNER
AS
$$
BEGIN
    -- Validate input: enforce maximum prompt length
    IF (LENGTH(:user_prompt) > 4000) THEN
        RETURN 'Error: Prompt exceeds maximum length of 4000 characters';
    END IF;

    -- Execute with owner's inherited rights
    -- System prompt constrains the model to data-related questions only
    LET system_prompt VARCHAR := 'You are a helpful data assistant. Only answer questions about data analysis, Snowflake, and business intelligence. Decline other topics. ';
    LET full_prompt VARCHAR := :system_prompt || :user_prompt;
    RETURN SNOWFLAKE.CORTEX.COMPLETE('snowflake.models.claude-4-sonnet', :full_prompt);
END;
$$;

-- Grant usage on the procedure to the restricted role
GRANT USAGE ON PROCEDURE restricted_cortex_complete_claude(VARCHAR) 
    TO ROLE IDENTIFIER($RESTRICTED_ROLE);

In [ ]:
-- 3.8.2 Test stored procedure with Restricted Role
-- The Restricted Role does not have CORTEX_USER and claude-4-sonnet is not on
-- the allow list. The procedure works because it executes with the owner's rights.
-- The system prompt constrains responses to data-related topics.

USE ROLE ACCOUNTADMIN;

USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);
USE WAREHOUSE IDENTIFIER($WAREHOUSE_NAME);
USE ROLE IDENTIFIER($RESTRICTED_ROLE);

-- Call the procedure - this should work despite role restrictions
CALL restricted_cortex_complete_claude(
    'What are the benefits of Snowflake Cortex?');

### 3.9 Secure Views for Batch Cortex Access

For batch use cases where users need to run Cortex Functions over table data, a **secure view** owned by the Admin role provides access without stored procedures. The view executes with the owner's privileges, so the restricted role only needs `SELECT` on the view — no `CORTEX_USER` required.

In [ ]:
-- 3.9.1 Create sample data and secure view as Admin Role
USE ROLE IDENTIFIER($ADMIN_ROLE);
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);
USE WAREHOUSE IDENTIFIER($WAREHOUSE_NAME);

-- Create sample data for the secure view demo
CREATE OR REPLACE TABLE SAMPLE_REVIEWS (
    review_id INT,
    review_text VARCHAR
);

INSERT INTO SAMPLE_REVIEWS VALUES
    (1, 'Snowflake Cortex makes AI accessible to all our analysts. Great product!'),
    (2, 'The query performance is terrible and the documentation is confusing.'),
    (3, 'Solid platform. Nothing exceptional but gets the job done reliably.'),
    (4, 'Love the seamless integration with our existing data pipelines.'),
    (5, 'Too expensive for what it offers. We are evaluating alternatives.');

-- Create a secure view that runs sentiment analysis using owner's privileges
CREATE OR REPLACE SECURE VIEW CORTEX_SENTIMENT_RESULTS AS
SELECT 
    review_id,
    review_text,
    SNOWFLAKE.CORTEX.SENTIMENT(review_text) AS sentiment_score
FROM SAMPLE_REVIEWS;

-- Grant SELECT on the secure view to the restricted role
GRANT SELECT ON VIEW CORTEX_SENTIMENT_RESULTS TO ROLE IDENTIFIER($RESTRICTED_ROLE);
GRANT SELECT ON TABLE SAMPLE_REVIEWS TO ROLE IDENTIFIER($RESTRICTED_ROLE);

In [ ]:
-- 3.9.2 Test secure view with Restricted Role
-- The restricted role can query sentiment results without CORTEX_USER access

USE ROLE ACCOUNTADMIN;

USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);
USE WAREHOUSE IDENTIFIER($WAREHOUSE_NAME);
USE ROLE IDENTIFIER($RESTRICTED_ROLE);

-- Query the secure view - sentiment analysis runs with the view owner's privileges
SELECT * FROM CORTEX_SENTIMENT_RESULTS
ORDER BY sentiment_score;

## 4. Fine-Tuning Models

Fine-tuning customizes LLMs using your own data. Training data must have `prompt` and `completion` columns.

### 4.1 Prepare Training Data

Training data must come from a Snowflake table/view with columns named `prompt` and `completion`.

In [ ]:
-- 4.1.1 Create training data table
-- Switch to Data Scientist role (required for fine-tuning)
USE ROLE IDENTIFIER($DATA_SCIENTIST_ROLE);
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);
USE WAREHOUSE IDENTIFIER($WAREHOUSE_NAME);

-- Create a table with training data
-- Format: prompt and completion pairs
CREATE OR REPLACE TABLE CORTEX_TRAINING_DATA (
    prompt VARCHAR,
    completion VARCHAR
);

-- Insert sample training data
INSERT INTO CORTEX_TRAINING_DATA (prompt, completion) VALUES
    ('What is Snowflake?', 'Snowflake is a cloud-based data platform that provides data warehousing, data lakes, data engineering, and data science capabilities.'),
    ('Explain Snowflake Cortex', 'Snowflake Cortex is a fully managed AI and ML service that brings LLMs and other AI capabilities directly to your data in Snowflake.'),
    ('What are Snowflake stages?', 'Stages in Snowflake are locations where data files are stored for loading and unloading data. They can be internal or external.'),
    ('How does Snowflake handle concurrency?', 'Snowflake uses multi-cluster warehouses and automatic scaling to handle concurrent queries without performance degradation.'),
    ('What is Time Travel in Snowflake?', 'Time Travel allows you to access historical data that has been changed or deleted within a defined period, typically up to 90 days.');

SELECT COUNT(*) AS training_records FROM CORTEX_TRAINING_DATA;


### 4.2 Create Fine-Tuned Model


In [ ]:
-- 4.2.1 Create a fine-tuned model
-- Note: Fine-tuning is currently available for specific models like mistral-7b
USE ROLE IDENTIFIER($DATA_SCIENTIST_ROLE);
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);

--DROP MODEL fine_tuned_model; -- if model already exists, otherwise comment out;
/*
SELECT SNOWFLAKE.CORTEX.FINETUNE(
  'CREATE',
  'fine_tuned_model_2',
  'mistral-7b',
  'SELECT prompt, completion FROM CORTEX_TRAINING_DATA'
);
-- Note: Fine-tuning is an asynchronous operation and may take time
*/

### 4.3 Monitor Fine-Tuning Progress


In [ ]:
-- 4.3.1 Check fine-tuning status
USE ROLE IDENTIFIER($DATA_SCIENTIST_ROLE);
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);

-- Show all fine-tuned models
SELECT SNOWFLAKE.CORTEX.FINETUNE('SHOW');

-- Get detailed information about a specific model
SELECT SNOWFLAKE.CORTEX.FINETUNE('DESCRIBE', '<your-fine-tune-job-id>'); -- Fine tune job id

### 4.4 Use Fine-Tuned Model for Inference


In [ ]:
-- 4.4.1 Use the fine-tuned model for inference
USE ROLE IDENTIFIER($ADMIN_ROLE);
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);
USE WAREHOUSE IDENTIFIER($WAREHOUSE_NAME);

-- Compare with base model
SELECT 
    'Fine-tuned Model' AS model_type,
    SNOWFLAKE.CORTEX.COMPLETE(
       'fine_tuned_model_2',
        'What is Snowflake Cortex?'
    ) AS response;

### 4.5 Grant Access to Fine-Tuned Models


In [ ]:
-- 4.5.1 Grant usage on fine-tuned model to other roles
USE ROLE IDENTIFIER($DATA_SCIENTIST_ROLE);
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);

-- Grant to Analyst role
GRANT USAGE ON MODEL fine_tuned_model_2 
    TO ROLE IDENTIFIER($ANALYST_ROLE);

In [ ]:
-- 4.5.2 Test access to fine-tuned model with Analyst Role

USE ROLE IDENTIFIER($ANALYST_ROLE);
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);
USE WAREHOUSE IDENTIFIER($WAREHOUSE_NAME);

-- Compare with base model
SELECT 
    'Fine-tuned Model' AS model_type,
    SNOWFLAKE.CORTEX.COMPLETE(
       'fine_tuned_model_2',
        'What is Snowflake Cortex?'
    ) AS response
UNION ALL
SELECT 
    'Base Model' AS model_type,
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        'What is Snowflake Cortex?'
    ) AS response;

---

## Additional Resources

- [Cortex Functions Overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/overview)
- [Cortex LLM Functions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/llm-functions)
- [Model Access Controls](https://docs.snowflake.com/user-guide/snowflake-cortex/aisql#supported-features)
- [Cross-Region Inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)
- [Fine-Tuning Models](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-finetuning)

---

**Version:** 2.0  
**Last Updated:** March 2026

---

## 5. Cleanup (Optional)

Run these commands to clean up all resources created in this runbook.

**Note:** Database and schema drops are commented out to prevent accidental deletion of existing resources.

In [ ]:
-- Cleanup: Drop all created objects
-- USE ROLE ACCOUNTADMIN;
-- Drop stored procedure
-- DROP PROCEDURE IF EXISTS restricted_cortex_complete_claude(VARCHAR);

-- Drop secure view and sample data
-- DROP VIEW IF EXISTS CORTEX_SENTIMENT_RESULTS;
-- DROP TABLE IF EXISTS SAMPLE_REVIEWS;

-- Drop training data table
-- DROP TABLE IF EXISTS CORTEX_TRAINING_DATA;

-- Drop fine-tuned model (if exists)
-- DROP MODEL IF EXISTS fine_tuned_model;

-- Drop warehouse
-- DROP WAREHOUSE IF EXISTS IDENTIFIER($WAREHOUSE_NAME);

-- Drop custom roles
-- DROP ROLE IF EXISTS IDENTIFIER($ADMIN_ROLE);
-- DROP ROLE IF EXISTS IDENTIFIER($DATA_SCIENTIST_ROLE);
-- DROP ROLE IF EXISTS IDENTIFIER($ANALYST_ROLE);
-- DROP ROLE IF EXISTS IDENTIFIER($RESTRICTED_ROLE);

-- Drop schema and database (commented out for safety)
-- DROP SCHEMA IF EXISTS IDENTIFIER($SCHEMA_NAME);
-- DROP DATABASE IF EXISTS IDENTIFIER($DATABASE_NAME);

-- Reset account-level parameters to defaults (optional)
-- ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'DISABLED';
-- ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'None';

-- Re-grant CORTEX_USER to PUBLIC if needed
-- GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE PUBLIC;